# Tutorial

Originally [from the Pyro intro](https://pyro.ai/examples/intro_long.html)

In [ ]:
%reset -s -f


In [ ]:
import logging
import os

import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import pyro

import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
import torch.nn as nn

from pyro.infer import MCMC, NUTS, RandomWalkKernel, HMC
from pyro.infer import Predictive

# Construct data

In [ ]:


# Set random seed for reproducibility
np.random.seed(42)

nobs = 40

# Generate data
x_obs = np.hstack([np.linspace(-0.2, 0.2, nobs // 2), np.linspace(0.6, 1, nobs // 2)])
noise = 0.02 * np.random.randn(x_obs.shape[0])
y_obs = x_obs + 0.3 * np.sin(2 * np.pi * (x_obs + noise)) + 0.3 * np.sin(4 * np.pi * (x_obs + noise)) + noise

x_true = np.linspace(-0.5, 1.5, 1000)
y_true = x_true + 0.3 * np.sin(2 * np.pi * x_true) + 0.3 * np.sin(4 * np.pi * x_true)

# Convert data to PyTorch tensors
x_train = torch.from_numpy(x_obs).float()
y_train = torch.from_numpy(y_obs).float()

# Set plot limits and labels
xlims = [-0.5, 1.5]
ylims = [-1.5, 2.5]

x_test = torch.linspace(xlims[0], xlims[1], 3000)

# Create plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_true, y_true, 'b-', linewidth=3, label="True function")
ax.plot(x_obs, y_obs, 'ko', markersize=4, label="Observations")
ax.set_xlim(xlims)
ax.set_ylim(ylims)
ax.set_xlabel("X", fontsize=30)
ax.set_ylabel("Y", fontsize=30)
ax.legend(loc=4, fontsize=15, frameon=False)

plt.show()


# Tutorial Neural Network Test

[From UvA DL Notebooks](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/DL2/Bayesian_Neural_Networks/dl2_bnn_tut1_students_with_answers.html)

In [ ]:



class MyFirstBNN(PyroModule):
    def __init__(self, in_dim=1, out_dim=1, hid_dim=5, prior_scale=10.):
        super().__init__()

        self.activation = nn.Tanh()  # or nn.ReLU()
        self.layer1 = PyroModule[nn.Linear](in_dim, hid_dim)  # Input to hidden layer
        self.layer2 = PyroModule[nn.Linear](hid_dim, out_dim)  # Hidden to output layer

        # Set layer parameters as random variables
        self.layer1.weight = PyroSample(dist.Normal(0., prior_scale).expand([hid_dim, in_dim]).to_event(2))
        self.layer1.bias = PyroSample(dist.Normal(0., prior_scale).expand([hid_dim]).to_event(1))
        self.layer2.weight = PyroSample(dist.Normal(0., prior_scale).expand([out_dim, hid_dim]).to_event(2))
        self.layer2.bias = PyroSample(dist.Normal(0., prior_scale).expand([out_dim]).to_event(1))

    def forward(self, x, y=None):
        x = x.reshape(-1, 1)
        x = self.activation(self.layer1(x))
        mu = self.layer2(x).squeeze()
        sigma = pyro.sample("sigma", dist.Gamma(.5, 1))  # Infer the response noise

        # Sampling model
        with pyro.plate("data", x.shape[0]):
            obs = pyro.sample("obs", dist.Normal(mu, sigma * sigma), obs=y)
        return mu

In [ ]:


model = MyFirstBNN()



In [ ]:
# Set Pyro random seed
pyro.set_rng_seed(42)

# Define Hamiltonian Monte Carlo (HMC) kernel
# NUTS = "No-U-Turn Sampler" (https://arxiv.org/abs/1111.4246), gives HMC an adaptive step size
nuts_kernel = NUTS(model, jit_compile=True)  # jit_compile=True is faster but requires PyTorch 1.6+

# Define MCMC sampler, get 50 posterior samples
mcmc = MCMC(nuts_kernel, num_samples=50)

# Run MCMC
mcmc.run(x_train, y_train)

In [ ]:


predictive = Predictive(model=model, posterior_samples=mcmc.get_samples())
preds = predictive(x_test)

In [ ]:
def plot_predictions(preds):
    y_pred = preds['obs'].T.detach().numpy().mean(axis=1)
    y_std = preds['obs'].T.detach().numpy().std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))
    xlims = [-0.5, 1.5]
    ylims = [-1.5, 2.5]
    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("X", fontsize=30)
    plt.ylabel("Y", fontsize=30)

    ax.plot(x_true, y_true, 'b-', linewidth=3, label="true function")
    ax.plot(x_obs, y_obs, 'ko', markersize=4, label="observations")
    ax.plot(x_obs, y_obs, 'ko', markersize=3)
    ax.plot(x_test, y_pred, '-', linewidth=3, color="#408765", label="predictive mean")
    ax.fill_between(x_test, y_pred - 2 * y_std, y_pred + 2 * y_std, alpha=0.6, color='#86cfac', zorder=5)

    plt.legend(loc=4, fontsize=15, frameon=False)

In [ ]:
plot_predictions(preds)

In [ ]:
class BNN(PyroModule):
    def __init__(self, in_dim=1, out_dim=1, hid_dim=10, n_hid_layers=5, prior_scale=5.):
        super().__init__()

        self.activation = nn.Tanh()  # could also be ReLU or LeakyReLU
        assert in_dim > 0 and out_dim > 0 and hid_dim > 0 and n_hid_layers > 0  # make sure the dimensions are valid

        # Define the layer sizes and the PyroModule layer list
        self.layer_sizes = [in_dim] + n_hid_layers * [hid_dim] + [out_dim]
        layer_list = [PyroModule[nn.Linear](self.layer_sizes[idx - 1], self.layer_sizes[idx]) for idx in
                      range(1, len(self.layer_sizes))]
        self.layers = PyroModule[torch.nn.ModuleList](layer_list)

        for layer_idx, layer in enumerate(self.layers):
            layer.weight = PyroSample(dist.Normal(0., prior_scale * np.sqrt(2 / self.layer_sizes[layer_idx])).expand(
                [self.layer_sizes[layer_idx + 1], self.layer_sizes[layer_idx]]).to_event(2))
            layer.bias = PyroSample(dist.Normal(0., prior_scale).expand([self.layer_sizes[layer_idx + 1]]).to_event(1))

    def forward(self, x, y=None):
        x = x.reshape(-1, 1)
        x = self.activation(self.layers[0](x))  # input --> hidden
        for layer in self.layers[1:-1]:
            x = self.activation(layer(x))  # hidden --> hidden
        mu = self.layers[-1](x).squeeze()  # hidden --> output
        sigma = pyro.sample("sigma", dist.Gamma(.5, 1))  # infer the response noise

        with pyro.plate("data", x.shape[0]):
            obs = pyro.sample("obs", dist.Normal(mu, sigma * sigma), obs=y)
        return mu

In [ ]:
# define model and data
model = BNN(hid_dim=10, n_hid_layers=5, prior_scale=5.)

# define MCMC sampler
nuts_kernel = NUTS(model, jit_compile=True)
mcmc = MCMC(nuts_kernel, num_samples=50,    warmup_steps=50)
mcmc.run(x_train, y_train)

In [ ]:
predictive = Predictive(model=model, posterior_samples=mcmc.get_samples())
preds = predictive(x_test)
plot_predictions(preds)

In [ ]:
print(mcmc.get_samples().keys())
mcmc.get_samples()["layers.0.weight"].shape

# Using a replicated infinite BNN class

### Build model and examine prior

In [ ]:
from mmbo.nn import TIBNN

In [ ]:
x_train2 = x_train.reshape((*x_train.shape, 1))
y_train2 = y_train.reshape((*y_train.shape, 1))
x_test2 = x_test.reshape((*x_test.shape, 1))

In [ ]:
# define model and data

model = TIBNN.from_data(x_train2, y_train2, width=1000, depth=1, output_bias=True,
                        bias_ordering=False, bias_positive=False)


In [ ]:
x_test2.shape

In [ ]:
num_plot = 50
#width = 100
#depth = 3

pltpoints = 1000

###########################

fig, axs = plt.subplots()


x = x_test2

#model = TIBNN(n_feat=1, n_out=1, width=width, depth=depth)

for _ in range(num_plot):
    y = model(x)
    axs.plot(x[:,0].flatten(), y.flatten())




In [ ]:
def plot_predictions2(preds):
    y_pred = preds['mu'].squeeze().T.detach().numpy().mean(axis=1)
    y_std = preds['mu'].squeeze().T.detach().numpy().std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))
    xlims = [-0.5, 1.5]
    ylims = [-1.5, 2.5]
    plt.xlim(xlims)
    plt.ylim(ylims)
    plt.xlabel("X", fontsize=30)
    plt.ylabel("Y", fontsize=30)

    ax.plot(x_true, y_true, 'b-', linewidth=3, label="true function")
    ax.plot(x_obs, y_obs, 'ko', markersize=4, label="observations")
    ax.plot(x_obs, y_obs, 'ko', markersize=3)
    ax.plot(x_test, y_pred, '-', linewidth=3, color="#408765", label="predictive mean")
    ax.fill_between(x_test, y_pred - 2 * y_std, y_pred + 2 * y_std, alpha=0.6, color='#86cfac', zorder=5)

    plt.legend(loc=4, fontsize=15, frameon=False)

### Sample from model using MCMC

In [ ]:
# Sample from Posterior using MCMC

# define MCMC sampler: a few options
nuts_kernel = NUTS(model, jit_compile=True, full_mass=False)
#rwm_kernel = RandomWalkKernel(model, init_step_size=0.01)

# Sample
mcmc = MCMC(nuts_kernel, num_samples=100, warmup_steps=100)
mcmc.run(x_train2, y_train2)

In [ ]:
print({k:mcmc.get_samples()[k].shape for k in mcmc.get_samples().keys()})
print(mcmc.get_samples()['_chain.1.scale'].mean())

In [ ]:
predictive = Predictive(model=model, posterior_samples=mcmc.get_samples(), return_sites=["obs","mu","_chain.0.bias"])
preds = predictive(x_test2)
print({k:preds[k].shape for k in preds})

In [ ]:
plot_predictions2(preds)

### Fit posterior using SVI

In [ ]:
def do_svi(svi, *args, check_interval=1000, improvement_threshold=.95, max_nonimproved = 3, **kwargs):
    losses = []
    current_loss = None
    nonimproved = 0
    while nonimproved < max_nonimproved:
        loss = svi.step(*args, **kwargs)
        losses.append(loss)
        if (len(losses)+1) % check_interval == 0:
            previous_loss = current_loss
            current_loss = sum(losses[-check_interval:])/check_interval
            if (previous_loss is not None) and (current_loss/previous_loss > improvement_threshold):
                nonimproved += 1
            else:
                nonimproved = 0
            logging.info("Step {} Average Elbo loss: {} {}".format(len(losses), current_loss, '*'*nonimproved))
    return losses

In [ ]:
# Fit posterior using SVI
logging.basicConfig(format='%(message)s', level=logging.INFO)

pyro.clear_param_store()

#auto_guide = model.guide()
diag_guide = pyro.infer.autoguide.AutoDiagonalNormal(model)
adam = pyro.optim.Adam({"lr": 0.001})
elbo = pyro.infer.Trace_ELBO()
svi = pyro.infer.SVI(model, diag_guide, adam, elbo)

losses = do_svi(svi, x_train2, y_train2, improvement_threshold=.99)

plt.figure(figsize=(5, 2))
plt.plot(losses)
plt.xlabel("SVI step")
plt.ylabel("ELBO loss");

In [ ]:
class AutoMVNfromDiag(pyro.infer.autoguide.AutoMultivariateNormal):

    def __init__(self, model, base_diag: pyro.infer.autoguide.AutoDiagonalNormal = None):
        super().__init__(model)
        # Clone the fitted parameters from the base DiagNormal guide
        if base_diag is not None:
            with torch.no_grad():
                self._base_diag_loc = base_diag.loc.clone()
                self._base_diag_scale = base_diag.scale.clone()
        else:
            self._base_diag_loc = None
            self._base_diag_scale = None

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        if self._base_diag_loc is not None:
            self.loc = pyro.nn.PyroParam(self._base_diag_loc)
        if self._base_diag_scale is not None:
            self.scale = pyro.nn.PyroParam(self._base_diag_scale, self.scale_constraint)

class AutoLRMVNfromDiag(pyro.infer.autoguide.AutoLowRankMultivariateNormal):

    def __init__(self, model, base_diag: pyro.infer.autoguide.AutoDiagonalNormal = None, rank = None):
        super().__init__(model, rank = rank)
        # Clone the fitted parameters from the base DiagNormal guide
        if base_diag is not None:
            with torch.no_grad():
                self._base_diag_loc = base_diag.loc.clone()
                self._base_diag_scale = base_diag.scale.clone()
        else:
            self._base_diag_loc = None
            self._base_diag_scale = None

    def _setup_prototype(self, *args, **kwargs):
        super()._setup_prototype(*args, **kwargs)
        if self._base_diag_loc is not None:
            self.loc = pyro.nn.PyroParam(self._base_diag_loc)
        if self._base_diag_scale is not None:
            self.scale = pyro.nn.PyroParam(self._base_diag_scale, self.scale_constraint)
            self.cov_factor = pyro.nn.PyroParam(torch.zeros_like(self.cov_factor))


In [ ]:
pyro.clear_param_store()

lrmvn_guide = AutoLRMVNfromDiag(model, diag_guide)
adam2 = pyro.optim.Adam({"lr": 0.0005})
elbo2 = pyro.infer.Trace_ELBO()
svi2 = pyro.infer.SVI(model, lrmvn_guide, adam2, elbo2)


In [ ]:
newlosses = do_svi(svi2, x_train2, y_train2, improvement_threshold = .99, max_nonimproved = 5)

plt.figure(figsize=(5, 2))
plt.plot(newlosses)
plt.xlabel("SVI step")
plt.ylabel("ELBO loss");

In [ ]:
svi_predictive = Predictive(model, guide=lrmvn_guide, num_samples=100)
svi_samples = svi_predictive(x_test2)
print({k:svi_samples[k].shape for k in svi_samples})

In [ ]:
plot_predictions2(svi_samples)

In [ ]:
svi_samples['_chain.1.scale'].mean(dim=[0,1])

In [ ]:
diag_guide

### Fit model using NeuTra reparameterized HMC

**Work in progress** (Will probably never be completed because the AutoIAFNormal guide is too big for ram unless the network is small/simple.)

In [ ]:
# Fit posterior using SVI
logging.basicConfig(format='%(message)s', level=logging.INFO)

pyro.clear_param_store()

auto_guide = pyro.infer.autoguide.AutoIAFNormal(model)
adam = pyro.optim.Adam({"lr": 0.005})
elbo = pyro.infer.Trace_ELBO()
svi = pyro.infer.SVI(model, auto_guide, adam, elbo)

losses = []
for step in range(5000):  # Consider running for more steps.
    loss = svi.step(x_train2, y_train2)
    losses.append(loss)
    if step % 500 == 0:
        logging.info("Elbo loss: {}".format(loss))

plt.figure(figsize=(5, 2))
plt.plot(losses)
plt.xlabel("SVI step")
plt.ylabel("ELBO loss");